# Baseline Models: Seasonal Naive + DeepAR + N-BEATSx

Establishes a performance lower bound before training transformer models.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.dpi'] = 120
HORIZON = 28
N_SERIES = 20
N_OBS = 200

# Synthetic M5-like data
rng = np.random.default_rng(42)
series_ids = [f'ITEM_{i:03d}' for i in range(N_SERIES)]
dates = pd.date_range('2014-01-01', periods=N_OBS)
rows = []
for sid in series_ids:
    base = rng.integers(5, 30)
    trend = rng.uniform(-0.02, 0.05)
    y = base + trend * np.arange(N_OBS) + 3 * np.sin(2*np.pi*np.arange(N_OBS)/7) + rng.poisson(2, N_OBS)
    for t, d in enumerate(dates):
        rows.append({'unique_id': sid, 'ds': d, 'y': max(0.0, y[t])})
df = pd.DataFrame(rows)

train_df = df[df.ds < '2014-06-16']
test_df  = df[df.ds >= '2014-06-16']
print(f'Train: {train_df.ds.min().date()} to {train_df.ds.max().date()}')
print(f'Test:  {test_df.ds.min().date()} to {test_df.ds.max().date()} (horizon={HORIZON})')

In [ ]:
# Seasonal Naive baseline
try:
    from statsforecast import StatsForecast
    from statsforecast.models import SeasonalNaive
    sf = StatsForecast(models=[SeasonalNaive(season_length=7)], freq='D', n_jobs=-1)
    sf.fit(train_df)
    naive_preds = sf.predict(h=HORIZON)
    print('SeasonalNaive trained OK')
    print(naive_preds.head())
except ImportError:
    print('statsforecast not installed — skipping')
    

In [ ]:
# N-BEATSx smoke (5 steps)
try:
    from ml.models.nbeatsx import NBEATSxForecaster, NBEATSxConfig
    cfg = NBEATSxConfig(horizon=HORIZON, input_size=56, max_steps=5, batch_size=4)
    model = NBEATSxForecaster(cfg)
    model.fit(train_df)
    nbeatsx_preds = model.predict()
    print('N-BEATSx smoke: OK')
    print(nbeatsx_preds.head())
except Exception as e:
    print(f'N-BEATSx skipped: {e}')

In [ ]:
# DeepAR smoke (5 steps)
try:
    from ml.models.deepar import DeepARForecaster, DeepARConfig
    cfg = DeepARConfig(horizon=HORIZON, input_size=52, max_steps=5, batch_size=4)
    model = DeepARForecaster(cfg)
    model.fit(train_df)
    deepar_preds = model.predict()
    print('DeepAR smoke: OK')
except Exception as e:
    print(f'DeepAR skipped: {e}')

In [ ]:
# Plot first series predictions vs actuals
sid = series_ids[0]
actuals = test_df[test_df.unique_id == sid].set_index('ds')['y']

fig, ax = plt.subplots(figsize=(12, 4))
train_s = train_df[train_df.unique_id == sid].set_index('ds')['y']
ax.plot(train_s.index[-60:], train_s.values[-60:], label='Train (last 60)', color='steelblue')
ax.plot(actuals.index, actuals.values, label='Actuals', color='black', lw=2)
ax.set_title(f'Baseline forecasts — {sid} (h={HORIZON})')
ax.legend()
plt.tight_layout()
plt.show()

## Results
- **Seasonal Naive**: strong baseline exploiting weekly periodicity (lag-7 repeat)
- **N-BEATSx**: CPU-only, interpretable trend/seasonal stacks, converges fast
- **DeepAR**: Normal distribution head enables PI; slower than N-BEATSx

→ Target: PatchTST + TFT should beat all three on sMAPE and PI coverage.